<!---
Notas de clase Métodos Computacionales
Por 
Óscar Antonio Restrepo Gutiérrez
--->

# Solving linear systems with numpy and scipy
1) [Real and complex matrices](#Matrices_reales_y_complejas).<br>
2) [Comparing matrices with python](#Comparación_de_matrices_con_python).<br>
3) [LU factorisation and special cases](#FactorizacionLU).<br>
4) [Diagonalisation](#Diagonalizacion). <br>
5) [Supplementary material](#Material_complementario).






NumPy and SciPy use optimised implementations of linear algebra libraries such as ATLAS, MKL, OpenBlas and LAPACK under the BLAS interface to carry out matrix operations; these libraries provide highly optimised routines for common linear algebra operations, so we don't implement our own and instead learn to use these. ATLAS, OpenBlas and MKL are implementations of BLAS, which do the basic addition, subtraction and multiplication operations, while LAPACK builds on BLAS and provides more advanced functions, such as LU, QR and Cholesky matrix decompositions; solving systems of linear equations; diagonalisation; etc.

Based on these libraries, NumPy provides efficient data structures and fundamental linear algebra operations; SciPy extends these capabilities with a broader set of functions. These libraries are imported as,
```python
import scipy.linalg 
import numpy.linalg
```

Here we'll use scipy:


In [ ]:
import numpy as np
import scipy.linalg as LA


<a id='Matrices_reales_y_complejas'></a> 
## 1) Real and complex matrices
A complex matrix is one that has complex numbers as entries, example, 

$$
\begin{equation}
A1 =  
\begin{pmatrix}
1   & 1-2j & j \\
1+2j&  1   & 0 \\
-j  &  0   & 2
\end{pmatrix}
\end{equation},
$$

The transpose matrix $A^t$ is obtained by swapping rows for columns, or $a_{ij}\Rightarrow a_{ji}$; in complex matrices, after transposing, you must also conjugate the entries, i.e., $a_{ij}\Rightarrow  a_{ji}^*$ (where $a^*_{ji}=x-yj$ is the conjugate of $a_{ji}=x+yj$); the conjugate transpose is defined as $A^\dagger$. In python the transpose (real or complex) is computed using the routine, 
```python 
numpy.transpose()
```    
If $A$ is complex you need to transpose and conjugate using,
```python
numpy.transpose().conj()
```    
But this is cumbersome; it's even easier to use, 
```python
A.T (transpose, use np.array() or np.matrix())
A.H (conjugate transpose, only works with np.matrix() )
```

Example:


In [ ]:
# Transpose of a real matrix
#A = np.arange(1,10).reshape(3,3)
A = np.array([[1,2,3],[4,5,6],[7,8,9]])
print ('A =\n',A)
print ('\nTranspose of A = \n',np.transpose(A)) # or also A.transpose()


In [ ]:
# Conjugate transpose of a complex matrix
A = np.matrix([[1+1j,2+2j,3+3j],[4+4j,5+5j,6+6j],[7+7j,8+8j,9+9j]])
#A = np.arange(1,10) + 1j*np.arange(1,10) # array: real + imag
#A = A.reshape(3,3)

print ('\nA =\n',A)
print ('\nConjugate transpose of A = \n',A.transpose().conj())
# or also
print ('\nTranspose A.H of A = \n',A.H)


### The dot product of complex vectors 
Defined by,
$$
\begin{align}
\mathbf{u}\cdot\mathbf{v}&=\mathbf{u}^\dagger\mathbf{v}\quad\longrightarrow\quad\text{gives a complex number},\\
\mathbf{u}\cdot\mathbf{u}&=\mathbf{u}^\dagger\mathbf{u}\quad\longrightarrow\quad\text{gives a real number},
\end{align}
$$

where $\mathbf{u}^\dagger$ is a row and $\mathbf{v}$ a column; also note that $\mathbf{v}^\dagger\mathbf{u}=(\mathbf{u}^\dagger\mathbf{v})^*$ gives the conjugate, i.e. it does not commute like the dot product for reals. The dot product of a vector with itself defines the squared norm $\mathbf{u}^\dagger\mathbf{u}=\|\mathbf{u}\|^2$ and always gives a real number; in python it's computed with,
```python
linalg.norm(u) # gives the square root of the dot product.
```
For the dot product $\mathbf{u}\cdot\mathbf{v}$ with numpy's `matmul()`, `dot()` and `@` methods, you always need to take the conjugate of $\mathbf{u}$:


In [ ]:
u = np.array([1,1+1j])
v = np.array([2+3j,1+1j])

u.conj().dot(v), np.dot(u.conj(), u), v.dot(v.conj()), # The norm must be real
#u.dot(v.conj()) # Changes with the conjugate.


In [ ]:
LA.norm(u)


### Symmetric and Hermitian matrices
A real matrix is symmetric if it equals its transpose: 
\begin{equation*}
A=A^t \mbox{  or   } a_{ij}=a_{ji}.
\end{equation*}
For complex matrices, a matrix is said to be Hermitian if it equals its conjugate transpose,
\begin{equation*}
A= A^\dagger \mbox{  or   } a_{ij}=a^*_{ji},
\end{equation*}
Note that the matrix $A1$ above is Hermitian, since its conjugate transpose gives $A1$ again. There are three ways to create a symmetric matrix, 
$$A^tA\quad \text{ or }\quad AA^t\quad \text{ or also }\quad \frac{A+A^t}{2},$$ 

this also works for Hermitian matrices, but the complex conjugate is taken.

Example:




In [ ]:
# Create a symmetric matrix
A = np.arange(1,10).reshape(3,3)

A@A.T, A.T@A, (A + A.T)*.5 # there are three ways


In [ ]:
# Create a complex matrix: real part plus imaginary part
n = 3  # dimension
A_re = np.random.randint(-10,10,size=(n,n)) # real part
A_im = np.random.randint(-10,10,size=(n,n)) # imaginary part
A = A_re + 1j*A_im

A # complex matrix


In [ ]:
# Create a Hermitian matrix, two ways:
# Note the diagonal is real
A@A.T.conj(), (A + A.T.conj())*.5


### Orthogonal and unitary matrices
For reals, if the transpose of a matrix equals its inverse, the matrix is said to be *orthogonal*, 

$$
\begin{equation*}
A^{-1}=A^t, \iff A^tA=I
\end{equation*}
$$

Properties of orthogonal matrices
+   its determinant is $\pm 1$,
+   the dot product of two vectors is preserved, $(A\mathbf{x})\cdot(A\mathbf{y})=\mathbf{x}^tA^tA\mathbf{y}=\mathbf{x}^t\mathbf{y}=\mathbf{x}\cdot \mathbf{y}=$ const,
+   the vectors formed by its columns are unit vectors and mutually orthogonal,
+   the change-of-basis matrix from one orthonormal basis (i.e., one whose vectors are unit vectors and mutually perpendicular) $\bf{\hat i, \hat j, \hat k}$ to another orthonormal basis $\bf{\hat i', \hat j', \hat k'}$ is orthogonal.

For example the matrix,

$$
\begin{equation}
A=\begin{pmatrix}
0&-0.80&-0.60\\0.80&-0.36&\;\;\,0.48\\0.60&\;\;\,0.48&-0.64
\end{pmatrix}
\end{equation}
$$

is orthogonal. For solving equations, note that if $A\mathbf{x}=\mathbf{b}$ then $\mathbf{x} = A^{-1}\mathbf{b}=A^t\mathbf{b}$, which simplifies the work. In general, an orthogonal transformation $\mathbf{x}'=A\mathbf{x}$ with determinant 1 is *proper* and can be interpreted as a rotation of the vector $\mathbf{x}$ into $\mathbf{x}'$ (if the determinant is -1 it is *improper* and is not a rotation but an inversion of the axes, i.e. like looking in a mirror; a simple swap of two columns or rows transforms $A$ from improper to proper).


For complex numbers, if the conjugate transpose of a matrix equals its inverse, the matrix is said to be *unitary*,

$$
\begin{equation*}
A^{-1}= A^\dagger \iff AA^\dagger=I,
\end{equation*}
$$

Properties of unitary matrices,
+  $|\det A|=1$ since $\det A=e^{j\phi}$ ($\phi$ any real number),
+  the dot product of two vectors satisfies $(A\mathbf{x})\cdot(A\mathbf{y})=\mathbf{x}^\dagger A^\dagger A\mathbf{y}=\mathbf{x}\cdot \mathbf{y}$,
+  its columns (or rows) form an orthonormal basis,
+  $A$ is diagonalisable,
+  its eigenvectors are orthonormal,
+  it can be written in exponential form $A = e^{jH}$, where $H$ is a Hermitian matrix,
+  and also $\det e^H = e^{\hbox{trace} (H)}$.

For example,

$$
\begin{equation}
A=\begin{pmatrix}
1/\sqrt 2  &1/\sqrt 2 &-0\\
-1/\sqrt 2j&1/\sqrt2 j&-0\\
0          & 0        & j
\end{pmatrix}
\end{equation}
$$

is unitary.

It's important to identify when you have this type of matrix, since for systems $A\mathbf{x}=\mathbf{b}$ you don't need to use libraries to find $\mathbf{x}$, given that the solution can be found easily analytically, since $\mathbf{x}=A^\dagger \mathbf{b}$.

**Exercise:** use python with the matrices above (and random vectors $\mathbf{x}, \mathbf{y}$ and $\mathbf{b}$) to verify the properties of orthogonal and unitary matrices (except for the diagonalisation properties, which will be discussed later in the last section).



In [ ]:
# Do the task


<a id='Comparación_de_matrices_con_python'></a> 
## Comparing matrices with python
When we compare integers we use `a == b`, but if they are floats we use `|a-b|< eps`, i.e. we compare with a certain tolerance, e.g. `eps = 1e-6`. 
In python you can compare matrices the same way but element by element; there are several functions for this and they depend on whether the matrices have integer or float values.
The most common functions for comparing matrices (and arrays) with integer-type coefficients are,
```python
(A == B)                 # gives a matrix of falses and trues
(A == B).all()           # gives false or true
np.equal(A, B)           # gives a matrix of falses and trues
np.array_equal(A, B)     # gives false or true
```
and if the coefficients are float type,
```python
(abs(A-B) < 1e-8)        # gives a matrix of falses and trues
(abs(A-B) < 1e-8).all()  # gives false or true
np.amax(abs(A-B)) < 1e-8 # gives false or true (amax returns the array's max element)
np.isclose(A, B)         # gives a matrix of falses and trues 
np.allclose(A, B)        # gives false or true
```   
Let's check whether $A$ is symmetric or Hermitian in the following cases:


In [ ]:
# Check whether A is symmetric
A = np.array([[1,2,3],
              [4,5,6],
              [7,8,9]])

(A.T == A).all()


In [ ]:
# Check whether A is Hermitian
A = np.matrix([[1, 1-2j, 1j],
               [1+2j, 1, 0],
               [-1j, 0, 2]])

(A.H == A).all() 
# A.H == A  # Check what happens if you remove .all()


In [ ]:
# Float comparison
A = np.random.rand(3,3)
B = A + A*1e-5

(abs(A-B) < 1e-4).all(), (abs(A-B) < 1e-6).all()


<a id='FactorizacionLU'></a> 
## LU factorisation and special cases
### General LU factorisation case
Suppose a matrix can be expressed as the product of two matrices $A = LU$, where $L$ is a lower triangular matrix and $U$ is an upper triangular matrix; if this is the case, it can be shown that the number of operations goes from $O(n^3/3)$ to $O(2n^2)$. If $A$ is factorised as $LU$ then,
$$
\begin{align}
  A\mathbf{x} & = \mathbf{b}\\
  L(U\mathbf{x})&= \mathbf{b}
\end{align}
$$
If we define $\mathbf{y}=U\mathbf{x}$ then the problem is solved as,
$$
\begin{align}
L\mathbf{y} &= \mathbf{b}\\
U\mathbf{x} &= \mathbf{y}.
\end{align}
$$
If $L$ and $U$ are not already known, these two matrices must first be computed; this operation has the same cost as doing a Gaussian elimination, i.e., $O(n^3/3)$. However, once $L$ and $U$ are known, the $LU$ decomposition can be applied to any value of $\mathbf{b}$ — this is where its importance lies. For example, imagine you have to compute $A\mathbf{x}_n=\mathbf{b}_n$ for $n=1,2,..., N$ (another typical case is the iterated method $A\mathbf{x}_{n+1}=\mathbf{x}_n$); this operation would take a time of $O(N*n^3/3)$, whereas if $LU\mathbf{x}_n=\mathbf{b}_n$ is used, $A=LU$ only needs to be computed once for $n=1$ (with an approximate time of $O(n^3/3)$); once $LU$ has been computed, the remaining steps only take a time of $O(2(N-1)n^2)$. 

Not every non-singular matrix can be decomposed in the $LU$ form, but if permutations are used it can be shown that $PA=LU$, where $P$ is the permutation matrix formed by swapping the rows of the identity matrix; for example,

$$
\begin{equation}
PA=\begin{pmatrix}
 0&1& 0\\
 1&0& 0\\
 0&0& 1
\end{pmatrix}
\begin{pmatrix}
 a&b& c\\
 A&B& C\\
 g&h& i
\end{pmatrix}
=\begin{pmatrix}
 A&B& C\\
 a&b& c\\
 g&h& i
\end{pmatrix}
=LU=\begin{pmatrix}   
   1      & 0      & 0 \\
   L_{21} & 1      & 0 \\
   L_{31} & L_{32} & 1\\
\end{pmatrix}
\begin{pmatrix}   
   U_{11} & U_{21} & U_{31} \\
        0 & U_{22} & U_{32} \\
        0 & 0      & U_{33}
\end{pmatrix} \\
\end{equation}
$$

Note that $L$'s diagonal consists of ones. In this case the factorisation has the form $A = P^{-1}LU = P^tLU$ ($P$ has the property that its inverse equals its transpose).
In scipy.linalg the routine that computes the factorisation is:

    P, L, U = scipy.linalg.lu(A)
The output is a tuple with 3 matrices.

Example:


In [ ]:
import numpy as np
import scipy.linalg as LA
M = np.array([[7, 3, -1, 2], 
           [3, 8, 1, -4], 
           [-1, 1, 4, -1], 
           [2, -4, -1, 6] ])

P, L, U = LA.lu(M)

print("\n P = \n",P)
print("\n L = \n",L)
print("\n U = \n",U)


The system $A\mathbf{x}=\mathbf{b}$ is solved in two steps: 1) compute the pivoted $LU$ decomposition of the matrix, which is $A = P^tLU$, and assign the value to a new variable,

     lu = scipy.linalg.lu_factor(A)
2) then solve the system for $lu$

     x = scipy.linalg.lu_solve(lu,b)
Example:



In [ ]:
# lu_factor returns the tuple lu = (LU, piv)
# LU:  matrix storing U in the upper triangle and L in the lower triangle 
#      (L's diagonal ones are not stored).
# piv: array, pivot indices representing the permutation matrix P:
#      row i of the matrix was swapped with row piv[i].

b  = np.array([1,1,1,1])
lu = LA.lu_factor(M)   
x  = LA.lu_solve(lu,b) # solve M x = b from lu_factor: lu_solve((LU,piv), b).
print ("Value using LU: x =",x)
print ("Using solve   : x =", LA.solve(M,b))


Note that since $L$ and $U$ are triangular and all of $L$'s elements are one, the determinant is simply the product of $U$'s diagonal elements, i.e., 

$$\det A = \det P^t\det L\det U = (-1)^k\det U = (-1)^k\prod_{i=1}^n U_{ii}.$$

where $P$'s determinant is $(-1)^k$ and $k$ is the number of permutations.


**Definition**: a matrix is strictly diagonally dominant when it satisfies,
$$
\begin{equation*}
|a_{i,i}| > \sum^n_{\substack{j=1\\ j\neq i}} |a_{i,j}|,\quad \forall i=\{1,...,n\}.
\end{equation*}
$$

In this case Gaussian elimination can be applied, and the system $A\mathbf{x}=\mathbf{b}$ has a stable solution and no row or column swaps are needed.

### Positive-definite matrix
A matrix is defined as positive if for any value of $\mathbf{x}$ it holds that $\mathbf{x}^tA\mathbf{x}>0$.
Properties:
+  $A$ has an inverse.
+  $a_{ii}>0$.
+  $(a_{i j})^2 < a_{ii}a_{jj}$ , for each  $i \neq j$.
+  The determinant of each of the leading principal minors is greater than zero.
+  Gaussian elimination can be performed without row or column swaps.
+  If $A$ is symmetric, a sufficient condition for it to be positive definite is that it has positive diagonal elements and is diagonally dominant.
+  If $A$ is symmetric, there is an invertible matrix $B$ such that $A=B^tB$ (or also $A=BB^t$), since $x^t\!Ax = (Bx)^t Bx > 0$ for $x\ne 0$. 

This last property helps us build positive-definite matrices from any matrix $B$; for example $B$ could be a matrix with random entries: `B=np.random.random(n,n)`. Note that not every positive-definite matrix needs to be symmetric — it only needs to satisfy $\mathbf{x}^tA\mathbf{x}>0$; for example

$$
(x\ y)\left(\matrix{1&1\cr -1&1}\right)\left(\matrix{x \cr y}\right)=(x\ y)\left(\matrix {x+y\cr-x+y}\right)=(x^2+xy)+(-xy+y^2)=x^2+y^2 > 0.
$$
Also note that if you use $A=(B+B^t)/2$, $A$ is not necessarily positive definite, even though it is symmetric; for example the matrix $B=\left(\matrix{1&2\cr 3&4}\right)$ gives, $(x\ y)\left(\matrix{1&5/2\cr 5/2&4}\right)\left(\matrix{x \cr y}\right)=x^2 + 4y^2 + 5xy$, which can be less than zero.

**Task**: a) build matrices $A$ from random matrices $B$ of various sizes $n$ and verify the properties above. b) if `B=np.arange(n*n).reshape(n,n)`, what do you conclude?


In [ ]:
# Do the task:


### Cholesky decomposition $A=LL^t$
In general, $A$ is a symmetric positive-definite matrix if and only if it can be decomposed in the form $LL^t$, i.e.

\begin{align}
A=LL^t & =
\begin{pmatrix}   
   L_{11} & 0      & 0 \\
   L_{21} & L_{22} & 0 \\
   L_{31} & L_{32} & L_{33}\\
\end{pmatrix}
\begin{pmatrix}   
   L_{11} & L_{21} & L_{31} \\
        0 & L_{22} & L_{32} \\
        0 & 0      & L_{33}
\end{pmatrix} \\
& =
\begin{pmatrix}   
   L_{11}^2     & L_{21}L_{11}              &  L_{31}L_{11}   \\
   L_{21}L_{11} & L_{21}^2 + L_{22}^2       & L_{31}L_{21}+L_{32}L_{22}\\
   L_{31}L_{11} & L_{31}L_{21}+L_{32}L_{22} & L_{31}^2 + L_{32}^2+L_{33}^2
\end{pmatrix}
\end{align}

This is known as the Cholesky decomposition or factorisation. If $A$ is not positive definite the decomposition is not unique. The Cholesky algorithm, which is a modified version of Gaussian elimination, can be used to compute the $LL^t$ matrix decomposition; the Cholesky matrix decomposition is done in scipy with the function,
```python   
scipy.linalg.cholesky()
```
and it can also be applied to complex matrices, i.e., if $A$ is Hermitian positive definite: $A=LL^\dagger$.

Example


In [ ]:
# in reals
M = np.array([[7, 3, -1, 2], 
              [3, 8, 1, -4], 
              [-1, 1, 4, -1], 
              [2, -4, -1, 6] ])
L = LA.cholesky(M, lower=True)
print ("Cholesky (real) L =\n",L)


# in complex numbers
A = np.array([[1,-2j],[2j,5]])
L = LA.cholesky(A, lower=True)
print ("\nCholesky (complex) L =\n",L)


**Exercise**: using python, verify that $M$ satisfies the properties of a positive-definite matrix.  

### Symmetric matrix $A = LDL^t$;
If $A$ is symmetric and positive definite it can also be decomposed in the form,

\begin{align}
A=LDL^t & =
\begin{pmatrix}   
   1      & 0 & 0 \\
   L_{21} & 1 & 0 \\
   L_{31} & L_{32} & 1\\
\end{pmatrix}
\begin{pmatrix}   
 D_1 & 0 & 0 \\
   0 & D_2 & 0 \\
   0 & 0 & D_3\\
\end{pmatrix}
\begin{pmatrix}
   1 & L_{21} & L_{31} \\
   0 & 1 & L_{32} \\
   0 & 0 & 1\\
\end{pmatrix} \\
& = \begin{pmatrix}   
   D_1       & L_{21}D_1                   & L_{31}D_1   \\
   L_{21}D_1 & L_{21}^2D_1 + D_2           & L_{31}L_{21}D_{1}+L_{32}D_2 \\
   L_{31}D_1 & L_{31}L_{21}D_{1}+L_{32}D_2 & L_{31}^2D_1 + L_{32}^2D_2+D_3
\end{pmatrix}.
\end{align}

As can be seen, $A$ is symmetric, $L$ is lower triangular with a diagonal of ones and $D$ is a diagonal matrix. The Cholesky computation (either the $LL^t$ or $LDL^t$ decomposition) takes a time of order $t \approx O(n^3/6)$, i.e., roughly half the time of a normal Gaussian elimination.
Important: note that the matrix $L$ in the $A = LDL^t$ decomposition is not the same as in $A = LL^t$, but it holds that

$$A = {L D L}^t = L D^{1/2} (D^{1/2})^t L^t = L  D^{1/2} (LD^{1/2})^t.$$

The $LDL^t$ decomposition is not yet implemented in *scipy*, but it can be done with the `cholesky()` routine; if $S$ is defined as the diagonal matrix containing the elements of the main diagonal of $L^{ch}$ in the Cholesky decomposition, then it can be shown that,

$$D = S^2\\ L=L^{ch}S^{-1}.$$

The implementation is:


In [ ]:
def ldl(A):
    A = np.asmatrix(A) # if it's an array, view it as a matrix 
    # check that A is symmetric or Hermitian
    if np.allclose(A.H, A): # (A.H == A).all()
        Lch  = np.linalg.cholesky(A)
        S    = np.diag(np.diag(Lch))
        Sinv = np.diag(1/np.diag(S))
        
        D = np.matrix(S.dot(S))
        L = np.matrix(Lch.dot(Sinv))      
            
        return L, D
    else:
        print("A must be symmetric or Hermitian.")
        return None, None

    
M = np.array([[7, 3, -1, 2], 
              [3, 8, 1, -4], 
              [-1, 1, 4, -1], 
              [2, -4, -1, 6] ])
 
L, D = ldl(M)
print("L =\n",L)
print("\nD = \n",D)

# recover the original matrix:
print ("\nLDL^t =\n", L*D*L.H)


### Tridiagonal and banded matrices

When $A$ is a tridiagonal matrix, the system $A\mathbf{x}=\mathbf{d}$ has the form,

$$
\begin{equation*}
a_i x_{i - 1}  + b_i x_i  + c_i x_{i + 1}  = d_i
\end{equation*}
$$

or in matrix representation,

$$
\begin{equation*}
\begin{pmatrix}
   {b_1} & {c_1} & {   }  & {   }  & { 0 } \\
   {a_2} & {b_2} & {c_2}  & {   }  & {   } \\
   {   } & {a_3} & {b_3}  & \ddots & {   } \\
   {   } & {   } & \ddots & \ddots & {c_{n-1}}\\
   { 0 } & {   } & {   }  & {a_n}  & {b_n} \\
\end{pmatrix}
\begin{pmatrix}
   {x_1 }  \\
   {x_2 }  \\
   {x_3 }  \\
   \vdots  \\
   {x_n }  \\
\end{pmatrix} 
=
\begin{pmatrix}
   {d_1 }  \\
   {d_2 }  \\
   {d_3 }  \\
   \vdots  \\
   {d_n }  \\
\end{pmatrix}
\end{equation*}
$$

If $A$ is tridiagonal, the $LU$ factorisation takes quite a simple form:
$L$ has ones on its main diagonal and zeros everywhere else except
on the diagonal immediately above the main diagonal. $U$ has its only
non-zero entries on the diagonal below the main diagonal ([see problem](#problema10)). Therefore this algorithm
only requires $(5n − 4)$ multiplications/divisions and $(3n − 3)$ additions or subtractions, which shows it's much faster than using standard methods, which are of order $\frac{n^3}{3}$ (if there are more non-zero diagonals, the matrix is called a *banded matrix*).

In physics these matrices are common in cases such as cubic splines in interpolation, the tight-binding approximation method in quantum mechanics, and the Crank-Nicolson method for solving partial differential equations, etc.

To solve the system of equations $A\mathbf{x} = \mathbf{d}$, python provides the method,

```python
scipy.linalg.solve_banded, 
```
let's see:


In [ ]:
import numpy as np
import scipy.linalg as LA
# Diagonals of the triangular matrix:
c = [0, 3, 3, 3, 5]    # Upper diagonal, first element must be zero
b = [8, 2, 8, 3, 9]    # Central diagonal 
a = [1, 1, 3, 3, 0]    # Lower diagonal, last element must be zero

d = [1, 1, 1, 1, 1]
# create an nxn tridiagonal matrix 
A = np.diag(c[1:], 1) + np.diag(b, 0) + np.diag(a[:-1], -1)

print ("Tridiagonal matrix A =\n", A )

# Solve Ax = d, using scipy.linalg.solve_banded. 
# ab[u + i - j, j] == a[i,j], where (u,l) are the number of diagonals 
# above and below the main one; for tridiagonal it's (1,1).
ab = np.array([c,b,a]) # 3x5 matrix
x = LA.solve_banded ((1,1),ab,d) # Ax = d <==> LUx = d
print ("\nSolution Ax = d with A tridiagonal:\n x =",x)
print ("\nSolution Ax = d using solve:\n x =",LA.solve(A,d))


In [ ]:
# Compare computation times for a random tridiagonal matrix
n = 100
D = np.random.rand(3,n)
d = np.ones(n)
A = np.diag(D[0,1:], 1) + np.diag(D[1], 0) + np.diag(D[2,:-1], -1)

%timeit LA.solve_banded ((1,1),D,d) # Ax = d <==> LUx = d
%timeit LA.solve(A,d)


<a id='Diagonalizacion'></a> 
## Diagonalisation
The diagonalisation problem is related to finding the solution to the system of equations of the form, $$A\mathbf{x}=\lambda \mathbf{x},$$ where $\lambda$ is a scalar; in this case we can't use the previous methods, since $\mathbf{b}=\lambda \mathbf{x}$ is also unknown.
Matrix diagonalisation consists of transforming a matrix $A$ into another diagonal matrix $D$ that has the same properties as the original matrix; i.e. diagonalisation is equivalent to transforming a system of equations into another set in which the matrix takes a canonical form, i.e. $$A=UDU^{-1}.$$
In other words, a matrix is said to be diagonalisable if there is a matrix $U$ such that $$D=U^{-1}AU,$$ where $D$ is diagonal; if this holds, $A$ is said to be similar to $D$. 
If matrix $U$ is orthogonal — which happens when $A$ is symmetric — physically, diagonalising the matrix can be interpreted as a rotation of its axes so that they line up with its eigenvectors. 

### Applications of diagonalisation
+ **Mathematics**: computing powers of matrices $A^k$ and, in general, functions of matrices $f(A)$, solving systems of differential equations, etc., since $f(A) =U f(D) U^{-1}.$ 
+ **Physics**: computing the inertia tensor, coupled oscillators, simple passive-element circuits, quantum mechanics (any quantity that can be measured in a physical experiment is associated with a Hermitian operator; for example, the energy operator is called the Hamiltonian and is represented by a Hermitian matrix. When you diagonalise the Hamiltonian, the main diagonal gives you the system's energies).
+ **Astronomy**: rotation of astronomical objects such as asteroids, planetary oblateness, [velocity dispersion](https://en.wikipedia.org/wiki/Velocity_dispersion) of groups of objects such as open clusters, globular clusters, galaxies or galaxy clusters (the velocity dispersion is measured relative to the mean velocity of a cluster, i.e. the radial velocities of the group's members are measured via spectroscopy; once the group's velocity dispersion is obtained, it can be used to derive the group's mass), etc.
+ **Chemistry**: the rate at which the concentration of a reactant changes is proportional to its concentration and the concentration of another reactant.
+ **Biology**: the rate at which predator and prey populations change is solved by a system of equations that must be diagonalised.

See more [here](https://www.researchgate.net/profile/Tadeusz-Ostrowski/post/What-are-the-applications-of-Diagonalization-of-a-matrix/attachment/59d6262379197b80779846df/AS%3A320696398352387%401453471387821/download/Applications+of+diagonalization.pdf).


<!---
The diagonalization of a matrix can be interpreted as a rotation of the axes to align them with the eigenvectors.
--->



### Diagonalisation procedure 
The diagonalisation of an $n\times n$ matrix $A$ over a real or complex field $\mathbb{K}^n$ is done in two steps:

1)   Define the characteristic equation $$f(\lambda)=\hbox{det}(A - \lambda I)=0,$$ 
this equation gives a degree-$n$ polynomial, 
$$f(\lambda) = (\lambda - \lambda_1)^{n_1}(\lambda - \lambda_2)^{n_2}\cdots (\lambda - \lambda_k)^{n_k}$$
where $k\le n$ and the roots $\lambda_i$ are the eigenvalues, used to build matrix $D$; also $n=\sum^k_{i=1}n_i$. For repeated roots we say there is "*degeneracy*" of order $n_i$, but if all $n_i=1$ there is no degeneracy and there are $n$ distinct roots.

2) Substitute each root into the matrix equation $(A-\lambda_i I)\mathbf{x}=0$ and solve for $\mathbf{x}$; 
if the eigenvalue is not degenerate, we obtain, 
$$\mathbf{x} = \alpha_1 \mathbf{u}_1,$$
where $\alpha_1$ is any value, but if there's degeneracy of order $n_i$ for eigenvalue $\lambda_i$, the vector $\mathbf{x}$ found must be allowed to decompose as a linear combination of $n_i$ linearly independent vectors, 
$$\mathbf{x} = \alpha_1 \mathbf{u}_1 + \alpha_2 \mathbf{u}_2 + \cdots + \alpha_{n_i} \mathbf{u}_{n_i},$$
with $\alpha_1, \alpha_2 \cdots, \alpha_{n_i}$ any values.
If this doesn't hold, non-zero columns $\mathbf{u}_{i}$ are missing to form $U$, and the system is not diagonalisable. The $\alpha_i$ values can be any number and disappear when the vector is normalised.
In a diagonalisable system you must find $n$ linearly independent, normalised eigenvectors $\mathbf{u}_i$, $i=1,2,\cdots ,n$ (${\bf\hat u}={\bf u}/\|{\bf u}\|$), which are used to build the columns of the matrix,

$$
U = \begin{pmatrix}
\mid & \mid & & \mid \\
{\bf\hat u}_{1} & {\bf\hat u}_{2} & \cdots & {\bf \hat u}_{n}\\
\mid & \mid & & \mid \\
\end{pmatrix},
$$

where the eigenvalues form the matrix,
$$
 \begin{align} 
 D = \begin{pmatrix}
 \lambda_1 & 0 & \cdots &0 \\ 0 & \lambda_2 & 0 & 0\\ \vdots & 0 & \ddots & \vdots \\ 0 & 0 & \cdots & \lambda_n 
 \end{pmatrix}
\quad\text{ and if }\lambda_i\neq 0\quad\Rightarrow\quad
 D^{-1} = 
 \begin{pmatrix}
 \frac{1}{\lambda_1} & 0 & \cdots &0 \\ 0 & \frac{1}{\lambda_2} & 0 & 0\\ \vdots & 0 & \ddots & \vdots \\ 0 & 0 & \cdots & \frac{1}{\lambda_n}
 \end{pmatrix} 
 \end{align}.
$$


<!---
#### Solving $\boldsymbol{Ax=b}$ systems with diagonalization
First, consider the case in which $A=D$ were diagonal, then solving a system of equations $x=D^{-1}b$ would be trivial, since the inverse of $D$ is,

$$
 \begin{align} 
 D = \begin{pmatrix}
 d_1 & 0 & \cdots &0 \\ 0 & d_2 & 0 & 0\\ \vdots & 0 & \ddots & \vdots \\ 0 & 0 & \cdots & d_n 
 \end{pmatrix}
\iff 
 D^{-1} = 
 \begin{pmatrix}
 \frac{1}{d_1} & 0 & \cdots &0 \\ 0 & \frac{1}{d_2} & 0 & 0\\ \vdots & 0 & \ddots & \vdots \\ 0 & 0 & \cdots & \frac{1}{d_n}
 \end{pmatrix} 
 \end{align}
$$ 

so multiplication by a vector $b$ only requires $n$ operations.

In general, if $A=UDU^{-1}$, diagonalization can be used to solve the matrix problem, since $Ax=b$ can be written as,

$$
\begin{align}
(UDU^{-1})x = &\, (UU^{-1})b\\
UD(U^{-1}x) = &\, U(U^{-1}b)\\
\end{align}
$$

if we define $x'=U^{-1}x$ and $b'=U^{-1}b$, we obtain the canonical (standard) form,

$$\boxed{Dx'= b'}$$

this last equation provides a way of canonizing (standardizing) a system into the simplest form by reducing the system from $n\times n$ parameters to just $n$, while preserving the properties of the original matrix. Finally note that,

$$x=UD^{-1}U^{-1}b.$$

It's important to note that if the $\lambda_i$ are very small, this solution for $x$ can have very large errors.
--->

Unfortunately this procedure via the characteristic equation is impractical to implement; in computing, the eigenvectors and eigenvalues of a matrix are computed using the [QR algorithm](https://en.wikipedia.org/wiki/QR_algorithm), considered one of the 10 most important algorithms of the twentieth century; the supplement shows how to do the [QR decomposition](#Descomposición_QR) of a matrix. Therefore we use the following scipy routine (also defined in numpy) which returns a tuple with the eigenvalues $e$ and the matrix $U$ of eigenvectors:

    e,U = scipy.linalg.eig(A)

there's also the routine that computes only eigenvalues
   
    scipy.linalg.eigvals(A)


In general if $A$ is diagonalisable:
+   $\det A = \det D = \prod\limits_{i=1}^k{\lambda_i^{n_i}}$.
+   The columns of $U$ are linearly independent and form a basis for $\mathbb{K}^n$.
+   If there's degeneracy, the eigenvectors have the additional freedom of rotation, i.e. any other combination of rotated vectors sharing the same eigenvalue is also an eigenvector of $A$.
+   $A$ is invertible, $A^{-1}= UD^{-1}U^{-1}$ and has eigenvalues $1/\lambda_i$, since recalling that $(AB)^{-1} = B^{-1} A ^{-1}$, then,
$$A^{-1} = (UDU^{-1})^{-1} = (U^{-1})^{-1}D^{-1}U^{-1} = UD^{-1}U^{-1}.$$
+   the eigenvectors of $A^{-1}$ are the same eigenvectors of $A$.
+   Powers of $A$ can be computed as: $A^k =U D^k U^{-1}$ where $D^k = $ diag$(\lambda^k_1,\lambda^k_2 ,...,\lambda^k_n)$. 
+   In general $f(A) =U f(D) U^{-1}$, where $f(D) = $ diag$(f(\lambda_1),f(\lambda_2) ,...,f(\lambda_n))$.
+   If $A$ is upper or lower triangular, its eigenvalues are the diagonal elements and its inverse has eigenvalues $1/\lambda_i$.
+   Adding the same value $\alpha$ to all diagonal elements of matrix $A$ doesn't change its eigenvectors, and the eigenvalues shift by $\alpha$: $A\mathbf{x}=\lambda \mathbf{x}\iff (A + \alpha I)\mathbf{x}=(\lambda + \alpha) \mathbf{x}$.

Diagonalisation is especially useful for Hermitian and symmetric matrices, since these matrices have the following properties:

Hermitian matrices:

+   The eigenvalues of a Hermitian matrix are real.
+   The matrix $U$ that diagonalises $A$ is unitary, $D=U^\dagger AU$. 
+   If $A$ is positive definite ($\mathbf{x}^\dagger A\mathbf{x}>0$) and Hermitian, its eigenvalues are positive.

Similarly for symmetric matrices over the reals: 

+   The eigenvalues of a symmetric matrix are real. 
+   The matrix $U$ that diagonalises $A$ is orthogonal, $D=U^t AU$. 
+   If $A$ is positive definite ($\mathbf{x}^t A\mathbf{x}>0$) and symmetric, its eigenvalues are positive.

For these cases it's better to use
  
    e,U = np.linalg.eigh(A)


**Example: rigid-body rotation**

We know that the rotation of a rigid body is defined by the moment of inertia $I$, and that this depends on the direction of rotation, so we can compute properties such as rotational energy or angular momentum. If the rotation isn't aligned with the principal axes of rotation, the moment of inertia will be a matrix known as the inertia tensor; diagonalising the inertia tensor means finding the principal axes of rotation (which will be the eigenvectors of $I$). In general the rotational kinetic energy is computed by the matrix operation  

$$E_k=\frac{1}{2}\boldsymbol{\omega} I \boldsymbol{\omega},$$ 

and the angular momentum as $$\mathbf{L}=I \boldsymbol{\omega},$$

where $\boldsymbol{\omega}$ is the angular velocity vector.
If we impose the condition that the angular momentum be parallel to the angular velocity $I\boldsymbol{\omega} = \lambda\boldsymbol{\omega}$, we get the diagonalisation problem $I_D=U^{-1}IU$; then $U$ gives us the transformation to the new reference frame where the moment of inertia will be a diagonal matrix $I_D$, which simplifies the calculations but doesn't affect the results —
for example, consider the change of basis to the primed system via the transformation given by $U$ for the following rod (see figure)

|<img src="../figures/Rotacion_ejes.png" alt="Drawing" style="width: 500px;"/>|
|:--:| 
| *Figure: Rotation of axes so they line up with the principal axes of rotation of a rotated rod's moment of inertia.*|

then for the moment of inertia we'll have

$$U^{t}\mathbf{L}=U^{t}IU(U^{t}\boldsymbol{\omega})\quad\Longrightarrow\quad \mathbf{L'}=I_D \boldsymbol{\omega}'.$$

where $\mathbf{L}'$ and $\boldsymbol{\omega}'$ will be the angular momentum and angular velocity relative to the primed system, and $U^{t}=U^{-1}$, since $I$ is symmetric. It can be shown that the energy doesn't change under the switch to the primed system, i.e. $E_k=E'_k$ (prove it — done similarly to the angular momentum case). 

**Exercise**: a) Use Python to compute the moment of inertia for a set of particles placed randomly on a rod of size $(1,3)$ cm with unit masses each, b) compute the angular momentum and kinetic energy for any vector $\boldsymbol{\omega}$, c) diagonalise the inertia tensor and compute $\mathbf{L'}, \boldsymbol{\omega'}$ and verify that $E_k=E'_k$ and that $\mathbf{L'} = I_D \boldsymbol{\omega}'$.
The inertia tensor of a set of masses $m_i$ with coordinates $\mathbf{r}_i=(x_i,y_i,z_i)$ relative to their centre of mass is represented by the symmetric matrix

$$
\begin{align*}
\mathbf{I} = \left(
\begin{array}{lll}
I_{xx} & I_{xy} & I_{xz}\\
I_{xy} & I_{yy} & I_{yz}\\
I_{xz} & I_{yz} & I_{zz}
\end{array}\right),
\end{align*}
$$

where

$$
\begin{align*}
I_{xx} &= \sum_i m_i(y_i^2 + z_i^2), & \quad I_{yy} &= \sum_i m_i(x_i^2 + z_i^2), & \quad I_{zz} &= \sum_i m_i(x_i^2 + y_i^2),\\
I_{xy} &= -\sum_i m_ix_iy_i, & \quad I_{yz} &= -\sum_i m_iy_iz_i, & \quad I_{xz} &= -\sum_i m_ix_iz_i.
\end{align*}
$$



In [ ]:
# Do the inertia moment task




**Diagonalisation example**

Consider the upper triangular matrix,
$$A = \begin{pmatrix} -1&-1&1\\ 0&-2&1\\ 0&0&-1\\ \end{pmatrix}$$
then the eigenvalues are computed via the characteristic equation,

\begin{align}
f(\lambda) =\,  & 
\begin{vmatrix}
 -1-\lambda&     -1   &    1\\
 0         &-2-\lambda&    1\\
 0&0 & -1 -\lambda
\end{vmatrix}
= (-1 -\lambda)^2(-2 - \lambda).
\end{align}

To compute the eigenvectors, we first substitute $\lambda_1=-2$, which gives the system of 2 equations,

\begin{align} x-y &=0\\ 
                z &=0\\ 
\end{align}

in which $x=y$, meaning we can choose $x$ (or $y$) with any value; so we set $x = y=\alpha$, giving the eigenvector,

$$
\mathbf{x} = \alpha \begin{pmatrix} 1\\ 1\\ 0\\ \end{pmatrix}=\alpha\mathbf{u}_1
$$

For $\lambda_2=-1$ there is degeneracy of order two, so we need to find an $\mathbf{x}=\beta\mathbf{u}_2 + \gamma\mathbf{u}_3$ that is a linear combination of the two remaining eigenvectors. Substituting into the matrix gives the system,

\begin{align} 0-y+z &=0\\ 
              0-y+z &=0\\
              0+0+0 &=0
\end{align}

which tells us only that $y=z$, and $x$ can be any value, so we choose $x=\beta$ and $y=z=\gamma$, giving, 

$$
\mathbf{x} = \begin{pmatrix}  x\\ y\\ z\\ \end{pmatrix}
             = \begin{pmatrix}\beta\\ \gamma\\ \gamma \\\end{pmatrix}
             = \beta  \begin{pmatrix}  1\\ 0\\ 0\\ \end{pmatrix}
             + \gamma \begin{pmatrix}  0\\ 1\\ 1\\ \end{pmatrix}
             =\beta\mathbf{u}_2 + \gamma\mathbf{u}_3.
$$

When the vectors are normalised, the arbitrary constants $\alpha, \beta$ and $\gamma$ disappear, 

$$
\mathbf{\hat u}_1 = \frac{1}{\sqrt2} \begin{pmatrix} 1\\ 1\\ 0\\ \end{pmatrix}, 
\mathbf{\hat u}_2 = \begin{pmatrix} 1\\ 0\\ 0\\ \end{pmatrix}, 
\mathbf{\hat u}_3 = \frac{1}{\sqrt2} \begin{pmatrix} 0\\ 1\\ 1\\ \end{pmatrix}.
$$

These three eigenvectors form the columns of matrix $U$, which diagonalises $A$. 
<!---
Es importante enfatizar que en el caso de degeneración, debido a la elección arbitraria de los valores $\alpha_i$, pueden haber otras alternativas para expresar los autovectores para crear la matriz $U$.
--->

**Examples**:


In [ ]:
# 1) General real diagonalisable matrix from the previous example
# M = np.matrix([[-1,2,1],[6,-1,0],[-1,-2,-1]])
M = np.matrix([[-1,-1,1],
               [0,-2,1],
               [0,0,-1]])
e,U = LA.eig(M)
print ("Matrix:\n",M)
print ("\nEigenvalues: \n",e)
print ("\nEigenvectors: \n",U)

D=np.diag(e,0)  # Create the diagonal matrix using the eigenvalues; this gives an "array", 
                # so the matrix multiplication must be done as: 
                #         np.dot(np.dot(U,D),LA.inv(U)))
D=np.asmatrix(D)# but if at least one element is a "matrix", you can do: 
print ("\nVerifying that UDU^-1 == A:\n",U*D*LA.inv(U))


In [ ]:
# 2) Diagonalising a Hermitian matrix 
H = np.array([[1, 1-2j, 1j],
              [1+2j, 1, 0],
              [-1j, 0, 2]])
e,U = LA.eig(H)
print ("\n\nHermitian matrix:\n",H) 
print ("\nReal eigenvalues: \n",e) 
print ("\nComplex eigenvectors:\n",U)


For the real matrix, the solution found by numpy matches the analytical one, but in general $U$ may differ from the analytical solution — why? (answer: re-read the diagonalisation properties).

For the Hermitian matrix the eigenvalues appear to have a complex part, but on closer inspection each eigenvector's complex part is of order $10^{-16}$, so they can be considered zero.

Let's reprint the result with only 3 significant figures:


In [ ]:
# printing with only 3 significant figures after the point:
print ("\n\nHermitian matrix:\n",   np.round(H,3)) # print with 3 figures
print ("\nReal eigenvalues: \n",  np.round(e,3)) # after the point.
print ("\nComplex eigenvectors:\n",np.round(U,3)) #


**Exercise**: create a symmetric matrix $A$ and a Hermitian matrix $H$, both with varying dimensions, $n=5,10,15$,
compute their eigenvectors and eigenvalues, then compute their inverses and determinants via diagonalisation; verify in each case that $D$ is similar to $A$ and $H$ (i.e. they have the same properties). 


In [ ]:
# do the task


**Example**: for the Hermitian matrix $H$ from the previous example, use python to a) compute the matrix $F=e^{3jH}$, b) show that $F$ is unitary.

Solution: recall that in general $f(A) =U f(D) U^{-1}$; also $U^{-1}=U^\dagger$, so:


In [ ]:
# a)
D = np.diag(np.exp(3j*e)) # diagonal matrix with H's eigenvalues
F = U@D@U.T.conj()        # UDU^t, since U is unitary because H is Hermitian
#F = U@D@LA.inv(U)        # UDU^-1 also works
np.round(F,3)


In [ ]:
# b) 
I = np.round(F@F.T.conj(),3)
I#.real 


**Example**: let's show that $\cos^2(H)+\sin^2(H)=I$ gives the identity matrix, for any matrix $H$:


In [ ]:
# Task: verify for other matrices of dimension n
I = U@np.diag( np.cos(e)**2+np.sin(e)**2 )@U.T.conj()
np.round(I)


Not every matrix is diagonalisable; for example consider the matrix,

\begin{align}
A & =
\begin{pmatrix}   
   2 & 1 \\
   0 & 2 
\end{pmatrix}
\end{align}

which has a single degenerate eigenvalue equal to 2. The eigenvectors are equal to $\mathbf{u_1}=\mathbf{u_2}=(1,0)$ and the diagonal matrix is,

\begin{align}
D = &
\begin{pmatrix} 
2&0\\
0&2
\end{pmatrix} 
= 2I.
\end{align}

Computing the product $UDU^{-1} = 2I \neq A$, we see that $A$ is not recovered, i.e. $A$ is not similar to $D$.

When solved in numpy this gives:


In [ ]:
A = np.matrix([[2,1],[0,2]])
e,U = LA.eig(A)
print ("Matrix:\n",A)
print ("\nEigenvalues: \n",e)
print ("\nEigenvectors: \n",U)

D=np.diag(e)     # create the diagonal matrix using the eigenvalues 
D=np.asmatrix(D) # convert "np.array" to "np.matrix"
print ("\n The matrix product UDU^-1 is not A: \n",U*D*LA.inv(U))      


There is only one degenerate eigenvalue equal to 2 and both eigenvectors are the same (since 4.44089210e-16 can be considered zero), so this system has no solution. We see, then, that if there is at least one row or column equal to zero in $U$, then $A$ is not diagonalisable. 



### Summary

Properties of special matrices:
   +   Transpose matrix $A^t$ and conjugate transpose $A^\dagger$;
   +   Symmetric matrix $A=A^t$ (over the reals); 
   +   Hermitian matrix $A= A^\dagger$ (conjugate transpose over complex numbers); 
   +   Orthogonal matrix $A^{-1}=A^t$ (reals);
   +   Unitary matrix $A^{-1}= A^\dagger$ (complex numbers).<br>

LU factorisation and special cases:
   +   General case: matrix  $PA = LU$;
   +   Positive-definite matrix $A = LL^t$;
   +   Symmetric matrix $A = LDL^t$;
   +   Tridiagonal matrix $A = LU$ (banded form).<br>

Diagonalisation: 
   +   System of equations $A\mathbf{x}=\lambda \mathbf{x};$
   +   Diagonalisation $A=UDU^{-1} \iff D=U^{-1}AU;$
   +   Functions of matrices $f(A) =U f(D) U^{-1}.$


# Exercises

**Task 1)**: Use numpy to show that $A^{-1}$ has eigenvalues $1/\lambda_i$ and the same eigenvectors as $A$ (use the matrices above).

**Task 2)**: Define any two matrices, one symmetric and another Hermitian, and verify the properties of these two matrices mentioned above. 

**Task 3)**: The matrix 
$$A = \begin{pmatrix} 2&0&0\\ 1&1&2\\ 1&-1&4\\ \end{pmatrix}$$

has an analytical solution (see Burden) with eigenvalues $\lambda_1 =3$ and $\lambda_2 = 2$ with multiplicity 2; the eigenvectors are $(0, 1, 1)$, $(0, 2, 1)$ and $(-2, 0, 1)$; compare against python's solution, use
```python
np.matrix([[2, 0, 0],[1, 1, 2,],[1, -1, 4]])
```
 Why do the eigenvectors differ in python?

**Task 4)**: Generate several random $n\times n$ matrices $M$ with integer $a_{ij}$ and check that indeed $PA=LU$ and $A=P^tLU$, use 
```python 
A = np.random.randint(1,10, size=(n,n)) # nxn matrix with random numbers between 1 and 10
```       
**Task 5)** Repeat the previous exercise for symmetric matrices, and check that $A=LL^t$ and that $A = LDL^t$. b) Use the `triu()` (or `tril()`) commands to create triangular matrices and multiply by their transpose; decompose the result with `scipy.linalg.cholesky()` — what do you conclude? 

**Task 6)**: An $n\times n$ banded matrix $A$ is one for which all its elements are zero outside a diagonal band whose range is determined by the number of non-zero lower diagonals $l$ and upper diagonals $u$, i.e. if $a_{ij}$ are the matrix's elements, then

 $$a_{ij}=0\quad {\mbox{if}}\quad j<i-l\quad {\mbox{ or }}\quad j>i+u;\quad l,u\geq 0.$$

a) Write python code that creates banded matrices in general form for given values $n,l,u$.
b) Modify the code to extract the $l$ lower diagonals and $u$ upper diagonals from a given matrix $A$ and store the non-zero elements in a matrix $B$ of dimensions $(k,n)$ with $k=l+u+1$ and components $b_{ij}$ such that it can be used in `scipy.linalg.solve_banded((l,u),B,d)` to solve $Ax=d$. For example, if $A$ is $6\times 6$ with $u =1$, $l =2$ then $B$ must have the form
$$
B=
\begin{pmatrix}
 *    &  a_{01} & a_{12}&  a_{23}&  a_{34}&  a_{45}\\
a_{00}&  a_{11} & a_{22}&  a_{33}&  a_{44}&  a_{55}\\
a_{10}&  a_{21} & a_{32}&  a_{43}&  a_{54}&       *\\
a_{20}&  a_{31} & a_{42}&  a_{53}&      * &       *\\
\end{pmatrix}
$$

Hint: consider the fact that 

$$b_{ij} = a_{jq} \quad\hbox{for}\quad i = 0,...,(k-1) \quad\hbox{and}\quad  j = 0,...,(n-1), $$

where $q = i+j-l.$

**Task 7)** Generate a positive-definite matrix and solve the system $A\mathbf{x}=\mathbf{b}$; compare the computation times for each of the $LU$, $LL^t$ and $LDL^t$ cases against the time given by the `solve(A,b)` command (so the timing is meaningful, for each case run a loop that repeats the operation about 500 times, and make a histogram of the times using `plt.his()`).

**Task 8)** When a tridiagonal matrix has the form, 
$$
A=\begin{pmatrix}a & b\\
c & a & b\\
 & \ddots & a & \ddots \\
 &  &   & \ddots &  \\
 &  &  & c & a
\end{pmatrix},
$$
it's known as a *Toeplitz matrix* and [it can be shown](https://doi.org/10.1016/S0024-3795(99)00114-7) that the eigenvalues are given by

$$\lambda_{k}=a+2\sqrt{bc}\cos\left[\frac{k\pi}{(n+1)}\right], \quad k=1\cdots n$$

Write a program that computes the relative error in the eigenvalues for an $n\times n$ matrix using `LA.eig(A)`
(Use $n=10, a=5, b=2$ and $c=3).$

**Task 9)** In physics we often only need to find the largest and smallest eigenvalue of a matrix.

  a) Generate a matrix $A$ of dimension (10,10) and check that it's diagonalisable.

  b) To find the eigenvector associated with the largest eigenvalue, generate a random vector $\mathbf{u}$ and normalise it $\mathbf{x} = \mathbf{u}/\|\mathbf{u}\|$, then using the following iterative procedure (use the relative error as the stopping condition, $e=\frac{\|\mathbf{x}_{n+1}-\mathbf{x}_n\|}{\|\mathbf{x}_{n+1}\|}$), 

$$
\mathbf{u}_{n+1} = A\mathbf{x}_n,
$$ 

show that $\|\mathbf{u}_n\|$ converges quickly to $A$'s largest eigenvalue; compute the eigenvector as,

$$\lambda_{max}=\frac{\mathbf{x}_n\cdot A\mathbf{x}_n}{\mathbf{x}_n\cdot \mathbf{x}_n}.$$
  
  c) To find the smallest eigenvalue, repeat the same procedure but use the inverse of $A$, i.e. 
  
$$
  \mathbf{u}_{n+1} = A^{-1}\mathbf{x}_n\\
  \lambda_{min}=\frac{\mathbf{x}_n\cdot A\mathbf{x}_n}{\mathbf{x}_n\cdot \mathbf{x}_n}
$$
  
$1/\|\mathbf{u}_n\|$ converges quickly to the smallest eigenvalue in absolute value (this is because $1/\lambda$ converges to $A^{-1}$'s largest eigenvalue, so $\lambda$ converges to $A$'s smallest eigenvalue).  
 
 Finally verify the answer by diagonalising the matrix with python.

**Task 10)**: The *Rayleigh quotient* technique gives an approximation of an eigenvector with its respective eigenvalue given an initial approximation $b_{0}$ of the eigenvector, and has cubic convergence; the iteration is defined by, 

$$
\mathbf{b}_{i+1}={\frac {(A-\lambda _{i}I)^{-1}\mathbf{b}_{i}}{\|(A-\lambda_{i}I)^{-1}\mathbf{b}_{i}\|}},$$

where,

$$
\lambda_{i+1}={\frac {\mathbf{b}_{i+1}\cdot A\mathbf{b}_{i+1}}{\mathbf{b}_{i+1}\cdot \mathbf{b}_{i+1}}}.
$$

Implement the python code using the Gauss-Jordan routine for computing the inverse; use the relative error as the stopping condition, $e=\frac{\|\mathbf{b}_{i+1}-\mathbf{b}_i\|}{\|\mathbf{b}_{i+1}\|}$.


<a id='problema10'></a> 
**Task 11)**: the $LU$ decomposition of a tridiagonal matrix is given by,
 
$$
\begin{equation*}
\begin{pmatrix}
   {b_1} & {c_1} & {   }  & {   }  & { 0 } \\
   {a_2} & {b_2} & {c_2}  & {   }  & {   } \\
   {   } & {a_3} & {b_3}  & \ddots & {   } \\
   {   } & {   } & \ddots & \ddots & {c_{n-1}}\\
   { 0 } & {   } & {   }  & {a_n}  & {b_n} \\
\end{pmatrix}
= 
\begin{pmatrix}
   1     &       & {   }  & {   }  & { 0 } \\
   {l_2} & 1     &        & {   }  & {   } \\
   {   } & {l_3} & 1      &        & {   } \\
   {   } & {   } & \ddots & \ddots & {   } \\
   { 0 } & {   } & {   }  & {l_n}  & 1 \\
\end{pmatrix}
\begin{pmatrix}
   {v_1} & {c_1} & {   }  & {   }  & { 0 } \\
   {   } & {v_2} & {c_2}  & {   }  & {   } \\
   {   } & {   } & {   }  & \ddots & {   } \\
   {   } & {   } &        & \ddots & {c_{n-1}}\\
   { 0 } & {   } & {   }  & {   }  & {v_n} \\
\end{pmatrix}
\end{equation*}
$$ 

where,

$$
\begin{eqnarray}
v_1 &=& b_1\\
l_k &=& a_k/v_{k−1}\\
v_k &=& b_k−l_kc_{k−1} \quad \hbox{with}\quad  k= 2, . . . , n
\end{eqnarray}
$$

Now, to solve the system $A\mathbf{x}=\mathbf{d}$, solve $L\mathbf{y}=\mathbf{d}$ as,

$$
\begin{eqnarray}
x_n &=& y_n/v_n\\
y_k &=& d_k−l_ky_{k−1}, \quad \hbox{with}\quad  k= 2, . . . , n
\end{eqnarray}
$$

and solve $U\mathbf{x}=\mathbf{y}$ as,

$$
\begin{eqnarray}
x_n &=& y_n/v_n\\
x_k &=& (y_k−c_kx_{k+1})/v_k, \quad \hbox{with}\quad  k=n−1, . . . ,1
\end{eqnarray}
$$

Implement the python code that computes $L,U$ and solves $A\mathbf{x}=\mathbf{d}$.

**Task 12)**: The Fredholm integral of the second kind is widely used in physics, for example in signal processing and in radiology for radiation transport phenomena; it's defined as,

$$u(x)=f(x)+\int _{a}^{b}K(x,t)u (t)\,dt,$$

where the kernel $K(x,t)$ and $a,b$ are given. To approximate the function $u$ we partition $a=x_0<x_1<...<x_m=b$, creating the variables $u_i=u(x_i)$, which gives the system of $m$ equations,

$$u(x_i)=f(x_i)+\int _{a}^{b}K(x_i,t)u (t)\,dt \quad \hbox{with}\quad  i=0, . . . ,m.$$ 

If we take $a = 0, b = 1, f (x) = x^2$, and $K(x, t) = e^{|x−t|}$: 

a) show that the linear system:

$$
\begin{eqnarray}
u(0) = f (0) + \frac{1}{2}[K(0, 0)u(0) + K(0, 1)u(1)],\\
u(1) = f (1) + \frac{1}{2}[K(1, 0)u(0) + K(1, 1)u(1)],
\end{eqnarray}
$$

can be solved with the trapezoidal rule; find $u_0,u_1$.<br>
b) Implement the trapezoidal rule and Simpson's rule for any $f,K,m$ on $[a,b]$ (use the `scipy.linalg` libraries for Gaussian elimination and also with $LU$ decomposition) — can it be solved with Gaussian quadrature?<br>
c) Plot $u(x)$ and $u'(x)$ on $[a,b]$ with steps of $10^{-1}$ and $10^{-4}$. <br>
d) When is it advisable to do cubic spline interpolation? Solve the system for steps of $10^{-1}$, interpolate, and compare against the solution with $10^{-4}$.<br>


**Task 13)**: An antisymmetric matrix is defined as,

$$\Omega = \frac{A-A^t}{2}$$

it can be shown that in three dimensions this is equivalent to the matrix that generates the cross product,

$$
\Omega=[\omega ]_{\times }={\begin{bmatrix}\,\,0&\!-\omega _{3}&\,\,\,\omega _{2}\\\,\,\,\omega _{3}&0&\!-\omega _{1}\\\!-\omega _{2}&\,\,\omega _{1}&\,\,0\end{bmatrix}}.
$$

In python, write a routine that generates the angular velocity from random matrices $A$, compute the tangential velocity $\mathbf{v}_{\perp}$ and verify that the following expression holds,

$$\mathbf{v}_{\perp}=\Omega\cdot\mathbf{r}=\boldsymbol{\omega} \times\mathbf{r},$$

for any random vector $\mathbf{r}$ (see section [Effect of multiplying a matrix by a vector](Classes_17_18_Linear_Algebra.ipynb#Efecto_matriz_por_un_vector)).

**Task 14)**: The following electrical circuit is described by the equations
$$
\begin{matrix}
-V_1 +R_1I_1+R_2(I_1-I_2) = 0.0 \\
R_2(I_2-I_1)+R_3I_2+R_4(I_2-I_3)=0.0 \\
R_4(I_3-I_2)+R_5I_3+V_2= 0.0 
\end{matrix}
$$

a) Find the currents $I_1, I_2, I_3$ if $R_1=1.1$ kΩ, $R_2=2.3$ kΩ, $R_3 = 1.5$ kΩ, $R_4 = 0.55$ kΩ, $R_5 = 1.6$ kΩ, $V_1 = 20$ V and $V_2=15$ V.
b) Plot $I_1, I_2, I_3$ as a function of $V_1$ on the interval $[5,30]$ V.

**Task 15)**: On a static bar of length 7.80m, 4 forces act: at one end $F_0=926$N at an angle of $90^\circ$, $F_1$ applied at the other end at an angle of $69.3^\circ$, $F_2$ applied at a distance of $1.50$m from the first end at an angle of $251.1^\circ$, and $F_3$ applied at a distance of $2.60$m from the other end at an angle of $303.4^\circ$.

a) Draw the force diagram and show that the sum of forces and torques gives the equations:<br>

Sum of vertical forces:
$$F_1 \sin 69.3° − F_2\sin 71.1° − F_3 \sin 56.6° + 926 = 0,$$

sum of horizontal forces:
$$F_1 \cos 69.3° − F_2 \cos 71.1° + F_3 \cos 56.6° = 0,$$

torques:
$$7.80F_1\sin 69.3° − 1.50F_2 \sin 71.1° − 5.20 F_3\sin 56.6° = 0.$$

b) Use python to compute the forces $F_1, F_2, F_3$. <br>
c) Vary $F_0$'s angle from $0$ to $180^\circ$ and plot $F_1, F_2, F_3$ as a function of the angle.<br>
d) Add three more forces at different angles and points and repeat the previous steps. 


In [ ]:
# Hint for part a)
# Artistic representation of forces on a bar (not to real scale)
# Draw arrow with arrow(x,y, dx,dy, **kwargs), starts at (x, y) ending at (x+dx, y+dy).

import matplotlib.pyplot as plt
from numpy import *

θ1=69.3*pi/180; θ2=71.1*pi/180; θ3=56.6*pi/180 
F = .1 # scale factor for arrows (forces).

plt.figure(figsize=(15, 3))
plt.xlim(-.2,1.2)
plt.ylim(-.2,.2)
plt.axis('off') # remove axis (lines with numbers on the x, y axes)

# Bar 
plt.hlines(0,-.1,1.1,color="k",lw=1) # reference frame
plt.hlines(0,1,0,color="chocolate",lw=10) # Bar
# Force F1  
plt.arrow(0, 0, 0, F, head_width=0.02, head_length=.02, fc='k', ec='k',lw=5)
plt.text(.0, 0.15, "926 N")
# Force F2
plt.arrow(1,0, F*cos(θ1), F*sin(θ1), head_width=0.02, head_length=.02, fc='k', ec='k',lw=5)
plt.text(1.02, 0.01, r"69.3$^o\quad\bf F_1$")
# Force F3
plt.arrow(0.3, 0,-F*cos(θ2), -F*sin(θ2), head_width=0.02, head_length=.02, fc='k', ec='k',lw=5)
plt.text(0.18, -0.05, r"71.1$^o\quad\bf F_2$")

plt.arrow(0.7, 0,F*cos(θ3), -F*sin(θ3), head_width=0.02, head_length=.02, fc='k', ec='k',lw=5)
plt.text(.75, -0.05, r"56.6$^o\quad\bf F_3$")

# Distances torques
plt.text(.15, 0.02, "1.50 m")
plt.text(.45, 0.02, "3.70 m")
plt.text(.85, 0.02, "2.60 m")
180-251.1, 360.0-303.4


**Task 16)**: Let $B=A^{-1}$ be the inverse of a triangular matrix $A$; the inverse $B$ can be computed with the following algorithm
```python
n = dimension of A;
B = zeros;
for i=1:n
    B(i,i) = 1/A(i,i);
    for j=1:i-1
        s = 0;
        for k=j:i-1
            s = s + A(i,k)*B(k,j);
        end
        B(i,j) = -s*B(i,i);
    end
end
```
a) Implement the code in python.<br>
b) One of the most effective ways to compute the inverse of a symmetric (or Hermitian) matrix is via Cholesky decomposition, $A=LL^\dagger$; use the previous routine to compute the inverse. Solve via $PA=LU$ decomposition and via Gauss-Jordan and compare the running times (note the times will depend on whether the routines used are pure python or scipy).<br>
c) Compute the determinant of the inverse (Hint: note that $\det(A) = \prod_{i=1}^{n}l_{ii}^{2}$, so $\det(B) = \prod_{i=1}^{n}l_{ii}^{-2}$).


<a id='Material_complementario'></a> 
# Supplementary material
A few additional definitions, methods and routines.


The following algorithm also computes the $LDL^t$ decomposition, but in $L$ the diagonal has entries other than one:


In [ ]:
def ldl(A):
    A = np.asmatrix(A) # if it's an array, view it as a matrix 
    # check that A is symmetric or Hermitian
    if np.allclose(A.H, A): # (A.H == A).all()
        S = np.diag(np.diag(A)) # matrix with A's diagonal elements
        Sinv = np.diag(1/np.diag(A)) # inverse of S
        D = np.matrix(S.dot(S))      # D = S^2
        
        Lch = np.linalg.cholesky(A)  
        L = np.matrix(Lch.dot(Sinv)) # L = Lch*S^-1 
        return L, D
    else:
        print("A must be symmetric or Hermitian.")
        return None, None

    
M = np.array([[7, 3, -1, 2], 
              [3, 8, 1, -4], 
              [-1, 1, 4, -1], 
              [2, -4, -1, 6] ])
 
L, D = ldl(M)
print("L =\n",L)
print("\nD = \n",D)

# recover the original matrix:
print ("\nLDL^t =\n", L*D*L.H)


### Outer product
Let $\mathbf {u} =\left(u_{1},u_{2},\dots ,u_{m}\right)$ and $\mathbf {v} =\left(v_{1},v_{2},\dots ,v_{n}\right)$ be any two vectors of dimension $m$ and $n$; the outer product is defined as $\mathbf {u} \mathbf {v}^t$, which gives the $m\times n$ matrix obtained from the product,

$$
\mathbf {u} \mathbf {v} ^t
={\begin{pmatrix}u_{1}\\u_{2}\\ \vdots\\u_{m}\end{pmatrix}}{\begin{pmatrix}v_{1}&v_{2}&\cdots&v_{n}\end{pmatrix}} ={\begin{pmatrix}u_{1}v_{1}&u_{1}v_{2}&\dots &u_{1}v_{n}\\u_{2}v_{1}&u_{2}v_{2}&\dots &u_{2}v_{n}\\\vdots &\vdots &\ddots &\vdots \\u_{m}v_{1}&u_{m}v_{2}&\dots &u_{m}v_{n}\end{pmatrix}}
$$

Note that the outer product is different from the dot product, $\mathbf {u}^t\mathbf {v}$; in python it's computed with the numpy method,
```python
np.outer(u, v, out=None)
```
or also,
```python
np.c_[u]*v # c_[u] creates a column vector
```
If $\mathbf {u}$ and $\mathbf {v}$ are complex, you first need to take the conjugate of $\mathbf {v}^\dagger$, `np.outer(u, v.conj())`.
Example:


In [ ]:
# Outer product
u = np.array([1,2,3])
v = np.array([2,2,2])
np.outer(u,v), np.outer(v,u) # outer does not commute


### Householder transformation
Let $\mathbf {u}$ be any unit vector; the Householder transformation is defined by,

$$P=I -2\mathbf {u}\mathbf {u}^t$$

note that $\mathbf {u}\mathbf {u}^t$ is a matrix given by the outer product;
it's easy to show (do it) that matrix $P$ is symmetric and orthogonal, $P=P^t=P^{-1}$.

Example:


In [ ]:
# Householder
u = np.random.rand(3)
u = u/np.dot(u,u)**.5
P = np.eye(3)-2.*np.outer(u,u)
# Note that P.T == P, and P@P.T == I i.e. P.T == inv(P)
P, P.T, P@P.T 


<a id='Descomposición_QR'></a> 
### QR decomposition 

The QR decomposition of a matrix is defined as the product of an orthogonal matrix $Q$ (unitary if $A$ is complex) and an upper triangular matrix $R$, i.e. $A= QR$; it can be implemented by several methods, via Gram-Schmidt orthogonalisation, Givens rotations, and Householder reflections.

#### Schwarz-Rutishauser method
The Schwarz-Rutishauser algorithm is a modification of the classic Gram-Schmidt orthogonalisation process, proposed by H. R. Schwarz.

#### Householder method
The Householder method, or Householder reflection, is a transformation that reflects space with respect to a given plane; the procedure is: 

Let ${\mathbf {x}}=(x_1,x_2,...x_n)$ be the first column vector taken from matrix $A$ such that its magnitude is ${\|\mathbf {x}}\| = |α|$ (where $\alpha$ is a scalar); if $A$ is real, $\alpha$ must take the opposite sign of $\mathbf{x}$'s first component $x_1$ to avoid loss of precision, but for complex numbers it's generalised as,

$$\alpha =-e^{i\arg x_{1}}\|\mathbf {x} \|$$


then define the vector $\mathbf {e} _{1}=(1 0 … 0)^t$ and carry out the operation,

$$
\begin{aligned}
\mathbf {u} &=\mathbf {x} -\alpha \mathbf {e} _{1},\\
\mathbf {v} &={\mathbf {u} \over \|\mathbf {u} \|},\\
Q&=I-2\mathbf {v} \mathbf {v} ^\dagger.
\end{aligned}
$$

Then consider the product $QA$, which gives the matrix,

$$
R_1\equiv Q_{1}A={\begin{pmatrix}\alpha _{1}&* &\dots &* \\0&&&\\\vdots &&A_1'&\\0&&&\end{pmatrix}},
$$

where we redefine $Q=Q_1$; this process can be repeated gradually for the submatrix $A_1′$ obtained by deleting $R_1$'s first row and column, and the previous operations are repeated starting from $A_1'$'s first column to obtain the Householder submatrix $Q′_2$ (note $Q′_2$ has one fewer dimension than $Q_1$); in general, after $k$ repetitions of the operations we obtain the $n\times n$ matrix,

$$
Q_{k}={\begin{pmatrix}I_{k-1}&0\\0&Q_{k}'\end{pmatrix}},
$$

where $I_{k-1}$ is the identity matrix of dimension $k-1$ and $Q'_k$ is the reduced Householder submatrix of dimension $(n-k)\times (n-k)$ obtained at step $k$ of the procedure. Finally, after $n-1$ iterations of this process, we obtain the upper triangular matrix,

$$
R=Q_{n}\cdots Q_{2}Q_{1}A$$

and the unitary matrix,

$$
Q=Q_{1}^\dagger Q_{2}^\dagger\cdots Q_{n}^\dagger,
$$

with the sought-after property $A = QR$ as $A$'s decomposition, since from Householder's definition recall that $Q_k^\dagger=Q_k$ and $Q_k^\dagger Q_k=I$, so $A$ is recovered; in scipy it's implemented in the routine,
```python 
scipy.linalg.qr()
```
(for more [see wikipedia](https://en.wikipedia.org/wiki/QR_decomposition)). Let's look at the implementation:


In [ ]:
# With reals the results match LA.rq(); with complex numbers
# it differs, but Q is orthogonal and R is triu.
n = 100
A = np.random.randn(n,n) + 1j*np.random.randn(n,n)

def QR(A):
    n = len(A)
    Ak = np.copy(A)
    Q = np.eye(n,n, dtype=complex) 
    
    for k in range(n-1):
        x = Ak[0:,0]  # Column 1 of Ak, dimension n-k.
        x[0] += np.exp(1j*np.angle(x[0]))*LA.norm(x) # α = -exp(-i arg(x0))|x|
        # Householder transformation 
        Qp = np.eye(n-k,n-k) - 2.*np.outer(x,x.conj())/LA.norm(x)**2.
        Qk = np.eye(n,n, dtype=complex) # Matrix Q at step k.
        Qk[k:,k:] = Qp# Reduced matrix Q' at step k. 
        Q = Q@Qk      # Recall that Qk.T == Qk, it is Hermitian.
        Ak = (Qp@Ak)[1:,1:] # A′ is obtained from Q1A by removing the first row and column.
        #print('Qk=\n',np.round(Qk,3))
        #print('Ak=\n',np.round(Ak,3))
        #print('Q=\n',np.round(Q,3))
    R = Q.T.conj()@A 
    return Q, R    


def QR1(A): # slightly slower for large matrices.
    n = len(A)
    R = np.copy(A)
    Q = np.eye(n,n, dtype=complex) 
    
    for k in range(n-1):
        x = np.copy(R[k:,k])   # Column 1 of Ak, dimension n-k.
        x[0] += np.exp(1j*np.angle(x[0]))*LA.norm(x) # α = -exp(-i arg(x0))|x|
        # Householder transformation 
        Qp = np.eye(n-k,n-k) - 2.*np.outer(x,x.conj())/LA.norm(x)**2.
        Qk = np.eye(n,n, dtype=complex) # matrix Q at step k.
        Qk[k:,k:] = Qp  # reduced matrix Q' at step k.                 
        Q = Q@Qk        # recall that Qk.T == Qk, it is Hermitian.
        R = Qk@R        # matrix A at step k.
    return Q, R    


# Schwarz-Rutishauser algorithm
def QR2(A, type=complex): 
 
     A = np.array(A, dtype=type)
     m,n = np.shape(A)
 
     R = np.zeros((n, n), dtype=type)
     Q = np.array(A, dtype=type)

     for k in range(n):
         for i in range(k):
             R[i,k] = np.transpose(Q[:,i]).dot(Q[:,k])
             Q[:,k] = Q[:,k] - R[i,k] * Q[:,i]
 
         R[k,k] = LA.norm(Q[:,k])
         Q[:,k] = Q[:,k]/R[k,k]
 
     return -Q, -R


Q, R = QR(A)
#Q, R = LA.qr(A)
I = Q@Q.T.conj()
#np.round(R,3),np.round(Q,3),np.round(I,3), LA.det(Q) # verification
(abs(Q@R - A) < 1e-13).all() # verify that QR = A


In [ ]:
%timeit QR1(A)
%timeit QR(A)


### QR diagonalisation method
Before the discovery of the [QR algorithm](https://en.wikipedia.org/wiki/QR_algorithm), diagonalising required computing the zeros of the characteristic equation, 
$$f(\lambda)=\hbox{det}(A - \lambda I)=0,$$
the problem is that for large $n$ this method is unstable in floating-point arithmetic, but in 1959 Francis discovered the QR method for diagonalisation; in its simplest form it is: let $A$ be a symmetric matrix that decomposes as the product of an orthogonal matrix $Q$ and an upper triangular matrix $R$; then we build the series,

$$
A_{k+1}=R_{k}Q_{k}=Q_{k}^{T}Q_{k}R_{k}Q_{k}=Q_{k}^{T}A_{k}Q_{k}=Q_{k}^{-1}A_{k}Q_{k}
$$

then all the $A_k$ are similar matrices, which means they have the same eigenvalues. The algorithm is numerically stable since it operates via orthogonal transformations; to build the algorithm we take $A_0=A$ and apply QR decomposition; the pseudocode is,

\begin{aligned} 
&\text{$A_0 = A$,}\\ 
&\text{for k = 1,2, ... }\\
&\quad\text{$A_{k-1} = Q_kR_k,\quad$ apply QR decomposition to $A_{k-1}$,}\\
&\quad\text{$A_k = R_kQ_k,\,\,\quad\,\,$ its diagonal converges to the eigenvalues,}\\
&\quad\text{$Q = Q_1,...,Q_k,$ its columns converge to the eigenvectors,}\\
&\text{end} 
\end{aligned}

As a stopping condition you can use $|\max(A_k-A_{k-1})|<\epsilon$ or also $|\sum_{i,j} ((a_{ij})_k-(a_{ij})_{k-1})|<\epsilon$; 
let's see the python implementation:


In [ ]:
# Simpler version: returns eigenvalues and eigenvectors.
A = np.random.rand(4,4)
A =  (A+A.T)*.5 # A must be symmetric (Hermitian).
steps = 100     # If Ak isn't diagonal, increase steps.

Ak= np.copy(A) 
Q = np.eye(len(Ak))
for i in range(steps): 
    Qk,R = LA.qr(Ak) # QR(Ak) 
    #print(np.round(Ak,3),'\n') 
    Ak = R@Qk   # Ak must have the same eigenvalues as A.
    Q  = Q@Qk   # Q is the product of the Qk.        

np.round(Ak,3), np.round(Q,3), LA.eig(A) # Ak gives the eigenvalues, Q gives the eigenvectors.


In [ ]:
# Full code: EIGH(A) returns eigenvalues and eigenvectors.
import numpy as np

def QR(A):
    n = len(A)
    Ak = np.copy(A)
    Q = np.eye(n,n, dtype=complex) 
    
    for k in range(n-1):
        x = Ak[0:,0]  # Column 1 of Ak, dimension n-k.
        x[0] += np.exp(1j*np.angle(x[0]))*np.linalg.norm(x) # α = -exp(-i arg(x0))|x|
        # Householder transformation 
        Qp = np.eye(n-k,n-k) - 2.*np.outer(x,x.conj())/np.linalg.norm(x)**2.
        Qk = np.eye(n,n, dtype=complex) # Matrix Q at step k.
        Qk[k:,k:] = Qp# Reduced matrix Q' at step k. 
        Q = Q@Qk      # Recall that Qk.T == Qk, it is Hermitian.
        Ak = (Qp@Ak)[1:,1:] # A′ is obtained from Q1A by removing the first row and column.
    
    R = Q.T.conj()@A 
    return Q, R    

def EIGH(A, eps=1e-14, steps=None):
    n = len(A)
    if steps==None: steps = n**4
    
    Ai = np.copy(A) 
    Q  = np.eye(n)
    for i in range(steps): 
        Qk,R = QR(Ai) 
        
        Ak = R@Qk   # Ak must have the same eigenvalues as A.
        Q  = Q@Qk   # Q is the product of the Qk.
        
        if np.abs((Ak - Ai).sum())<eps: break # Stopping condition.
        Ai = Ak
    print("Steps:", i)    
    return np.diag(np.round(Ak,8)), np.round(Q,8) # Ak gives the eigenvalues, Q gives the eigenvectors.

# Test
n = 3
A = np.random.rand(n,n) # + 1j*np.random.rand(n,n)
A =  (A+A.T.conj())*.5 # A must be symmetric (Hermitian).
EIGH(A), np.linalg.eig(A)


### How to build random matrices that are orthogonal or unitary
The Householder method can be used to create an orthogonal matrix; let's see the implementation of an orthogonal matrix:


In [ ]:
# Create an orthogonal matrix with Householder:
def rvs(n=3): # Orthogonal matrix
    Q = np.eye(n,n) 
    
    for k in range(n-1):
        x = np.random.randn(n-k)
        x[0] += (np.exp(1j*np.angle(x[0]))*LA.norm(x)).real # α = -exp(-i arg(x0))|x|
        # Householder transformation 
        Qp = np.eye(n-k,n-k) - 2.*np.outer(x,x.conj())/LA.norm(x)**2.
        Qk = np.eye(n,n) # Reduced matrix Q' at step k.
        Qk[k:,k:] = Qp   # Matrix Q at step k.
        Q = Q@Qk         # Recall that Qk.T == Qk, it is Hermitian.
    return Q    

A = rvs(3)   
LA.det(A), A@A.T  # Gives the identity matrix and det


These methods are already implemented in scipy; let's see: 


In [ ]:
# Create a unitary matrix with Householder:
def rvsu(n=3): # unitary matrix
    Q = np.eye(n,n, dtype=complex) 
    
    for k in range(n-1):
        x = np.random.randn(n-k) + 1j*np.random.randn(n-k)
        x[0] += np.exp(1j*np.angle(x[0]))*LA.norm(x) # α = -exp(-i arg(x0))|x|
        # Householder transformation 
        Qp = np.eye(n-k,n-k) - 2.*np.outer(x,x.conj())/LA.norm(x)**2.
        Qk = np.eye(n,n, dtype=complex) # Reduced matrix Q' at step k.
        Qk[k:,k:] = Qp                  # Matrix Q at step k.
        Q = Q@Qk      # Recall that Qk.T == Qk, it is Hermitian.
    return Q    
    
U=rvsu(3)
abs(LA.det(U)), np.round(U@U.T.conj(),3)


In [ ]:
# Create orthogonal and unitary matrices with scipy: 
from scipy.stats import ortho_group 
from scipy.stats import unitary_group

A = ortho_group.rvs(3)   # Orthogonal matrix.
U = unitary_group.rvs(3) # Unitary matrix.
A@A.T, U@U.conj().T      # Verification.


### Creating random matrices with predetermined eigenvalues
First a random matrix $S$ is built with predetermined eigenvalues $D=\,$diag$\{\lambda_1,\lambda_2,...,\lambda_n\}$, by doing the operation $S=ADA^{-1}$, where $A$ is any random matrix. If $S$ needs to be symmetric then $A$ must be orthogonal (or unitary if $S$ needs to be Hermitian); in that case use `svr(n)` to create $A$ (this routine exists in scipy). In general $S$ will have the eigenvalues predefined by $D$.


In [ ]:
# Create a random matrix with predefined eigenvalues,
#     D = diag{λ1,λ2,...,λn}:

A = np.random.rand(4,4) # If A is random then,
D = np.diag([2,4,5,6])  # Create diagonal matrix, then.
S = A@D@LA.inv(A)       # Random matrix with given eigenvalues. 
LA.eig(S)               # From D, let's see: 


In [ ]:
# Create a random symmetric matrix with predefined eigenvalues,
#    D = diag{λ1,λ2,...,λn}:

A = rvs(4)          # For S to be symmetric, A must be orthogonal.
D = np.diag([2,4,5,6])  # Create diagonal matrix, with desired values.
S = A@D@A.T             # Random matrix with desired eigenvalues.
LA.eig(S), S            # Verify. 


In [ ]:
# Create a random Hermitian matrix with predefined eigenvalues,
#    D = diag{λ1,λ2,...,λn}:

U = rvsu(4)              # U must be unitary.
D = np.diag([2,4,6,8])   # Desired eigenvalues.
S = U@D@U.T.conj()       # Hermitian matrix with desired eigenvalues.
LA.eig(S), np.round(S,3) # Verify. 


In [ ]:
# Another version: create an orthogonal matrix with Householder.

# https://stackoverflow.com/questions/38426349/how-to-create-random-orthonormal-matrix-in-python-numpy
import numpy as np    

def rvs(dim=3):
    H = np.eye(dim)
    D = np.ones((dim,))
    for n in range(1, dim):
        x = np.random.randn(dim-n+1)
        D[n-1] = np.sign(x[0])
        x[0] -= D[n-1]*np.sqrt((x*x).sum())
        # Householder transformation
        Hx = np.eye(dim-n+1) - 2.*np.outer(x, x)/(x*x).sum()
        mat = np.eye(dim)
        mat[n-1:, n-1:] = Hx
        H = np.dot(H, mat)
        # Fix the last sign such that the determinant is 1
    D[-1] = (-1)**(1-(dim % 2))*D.prod()
    # Equivalent to np.dot(np.diag(D), H) but faster, apparently
    H = (D*H.T).T
    return H

A = rvs(dim=3)   
LA.det(A), A@A.T,  # Gives the identity matrix and det


In [ ]:
# Artistic representation of the rotation of axes in 2D
# so they line up with the rotation axes of a bar.

θ = 30*pi/180 #  Angle in degrees to radians
r = 1  # scale factor.

# ---------------------------------------------------------------
import matplotlib.pyplot as plt
from numpy import *
from matplotlib import patches


f, ax = plt.subplots(figsize=(10, 10))
#plt.figure(figsize=(10, 10))
plt.xlim(-1.1*r/2.1, 1.1*r)
plt.ylim(-1.1*r/2.2, 1.1*r)
plt.axis('off') # remove axis (lines with numbers on the x, y axes)


def R(θ): # Rotation matrix by angle θ 
    return  r*array([[cos(θ), sin(θ)],
                     [-sin(θ), cos(θ)]])

# 1) Draw bar
A = R(θ)
plt.arrow(-A[0,0]/2,-A[0,1]/2,A[0,0],A[0,1], head_width=0.,  head_length=.0, fc='c', ec="chocolate",lw=15)

# 2) Draw rotation arc 
plt.annotate(r'$\bfω$', xy=(.48, 0.34), fontsize='xx-large')
patch = patches.Arc(xy=(.5, 0.3), width=0.2, height=0.1, angle=120, theta1=0, theta2=280)
plt.arrow(.45 ,0.385, 0.001, 0.0001, head_width=0.02,  head_length=.02, fc='k', ec='k',lw=1)
ax.add_patch(patch)

# 3) Draw angle arc
a = linspace(0, θ, 20)
xa = 0.3*r*cos(a)
ya = 0.3*r*sin(a)
plt.plot(xa, ya, c='r', lw=3)
plt.annotate('θ', xy=(.2, 0.02), fontsize='xx-large')


# 4) Rotated axis system
plt.arrow(0, 0, A[0,0],A[0,1], head_width=0.02,  head_length=.02, fc='b', ec='b',lw=2)
plt.arrow(0, 0, A[1,0],A[1,1], head_width=0.02,  head_length=.02, fc='b', ec='b',lw=2)
# Axis labels
A = R(θ - 4*pi/180)
plt.text(  .9*A[0,0], .9*A[0,1], r"$x'$", fontsize='xx-large', c='b')
plt.text( 1.2*A[1,0], .8*A[1,1], r"$y'$", fontsize='xx-large', c='b')

# 5) Original axis system
A = R(0)
plt.arrow(0, 0, A[0,0],A[0,1], head_width=0.02, head_length=.02, fc='k', ec='k',lw=2)
plt.arrow(0, 0, A[1,0],A[1,1], head_width=0.02,  head_length=.02, fc='k', ec='k',lw=2)
# Axis labels
plt.text( .9,       -.1,       r"$x$", fontsize='xx-large', c='k')
plt.text(-.1,        .9,       r"$y$", fontsize='xx-large', c='k')

plt.savefig("../figures/Rotacion_ejes.png",transparent=True,format='png')



# References:
https://mathoverflow.net/questions/131527/eigenvalues-of-symmetric-tridiagonal-matrices

Computing a tridiagonal matrix
https://www.webpages.uidaho.edu/~barannyk/Teaching/LU_factorization_tridiagonal.pdf

How to generate a random unitary matrix:
http://home.lu.lv/~sd20008/papers/essays/Random%20unitary%20[paper].pdf

The QR Algorithm
https://people.inf.ethz.ch/arbenz/ewp/Lnotes/chapter4.pdf<br>
https://pi.math.cornell.edu/~web6140/TopTenAlgorithms/QRalgorithm.html<br>
Schwarz-Rutishauser Algorithm https://towardsdatascience.com/can-qr-decomposition-be-actually-faster-schwarz-rutishauser-algorithm-a32c0cde8b9b<br>
https://people.inf.ethz.ch/gander/papers/qrneu.pdf

Matrix examples
https://www.intmath.com/matrices-determinants/6-matrices-linear-equations.php


